# Glassnode x ARK: Decentralization Research Pipeline

**Phase 01 — Scaffolding & Mock Data**

This notebook provides the data pipeline boilerplate for the joint Glassnode x ARK Investment
decentralization report, comparing **Bitcoin (BTC)**, **Ethereum (ETH)**, and **Solana (SOL)**.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import requests
import os
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# API Keys (load from environment variables — never hardcode)
GLASSNODE_API_KEY = os.getenv("GLASSNODE_API_KEY", "YOUR_API_KEY_HERE")

In [ ]:
class DecentralizationVisualizer:
    """
    Reusable Plotly visualization engine for the Glassnode x ARK decentralization report.
    Supports line, bar, and pie charts with consistent styling and multi-asset color coding.
    """

    COLORS = {
        "BTC": "#F7931A",
        "ETH": "#627EEA",
        "SOL": "#14F195",
        "neutral": "#888888"
    }

    def __init__(self, theme: str = "light"):
        self.theme = theme
        self.template = "plotly_white" if theme == "light" else "plotly_dark"
        self.layout_defaults = dict(
            template=self.template,
            font=dict(family="Arial, sans-serif", size=12),
            title_font=dict(size=16, color="#333333"),
            legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
            margin=dict(l=60, r=60, t=80, b=60),
            hovermode="x unified"
        )

    def _get_color(self, name: str) -> str:
        return self.COLORS.get(name, self.COLORS["neutral"])

    def line_chart(self, df: pd.DataFrame, title: str, x_col: str,
                   y_cols: list, y_labels: list = None,
                   dual_axis: bool = False) -> go.Figure:
        if y_labels is None:
            y_labels = y_cols

        if dual_axis and len(y_cols) == 2:
            fig = make_subplots(specs=[[{"secondary_y": True}]])
            fig.add_trace(
                go.Scatter(x=df[x_col], y=df[y_cols[0]], name=y_labels[0],
                           line=dict(color=self._get_color(y_cols[0]))),
                secondary_y=False
            )
            fig.add_trace(
                go.Scatter(x=df[x_col], y=df[y_cols[1]], name=y_labels[1],
                           line=dict(color=self._get_color(y_cols[1]))),
                secondary_y=True
            )
            fig.update_yaxes(title_text=y_labels[0], secondary_y=False)
            fig.update_yaxes(title_text=y_labels[1], secondary_y=True)
        else:
            fig = go.Figure()
            for col, label in zip(y_cols, y_labels):
                fig.add_trace(
                    go.Scatter(x=df[x_col], y=df[col], name=label,
                               line=dict(color=self._get_color(col)))
                )

        fig.update_layout(title=title, xaxis_title=x_col, **self.layout_defaults)
        return fig

    def bar_chart(self, df: pd.DataFrame, title: str, x_col: str,
                  y_col: str, color_col: str = None) -> go.Figure:
        if color_col and color_col in df.columns:
            groups = df[color_col].unique()
            fig = go.Figure()
            for group in groups:
                subset = df[df[color_col] == group]
                fig.add_trace(
                    go.Bar(x=subset[x_col], y=subset[y_col], name=str(group),
                           marker_color=self._get_color(str(group)))
                )
            fig.update_layout(barmode="group")
        else:
            fig = go.Figure(
                go.Bar(x=df[x_col], y=df[y_col],
                       marker_color=self.COLORS["neutral"])
            )

        fig.update_layout(title=title, xaxis_title=x_col, yaxis_title=y_col,
                          **self.layout_defaults)
        return fig

    def pie_chart(self, labels: list, values: list, title: str) -> go.Figure:
        colors = [self._get_color(label) for label in labels]
        fig = go.Figure(
            go.Pie(labels=labels, values=values, hole=0.4,
                   marker=dict(colors=colors),
                   textinfo="label+percent", textposition="outside")
        )
        fig.update_layout(title=title, **self.layout_defaults)
        return fig

    def show(self, fig: go.Figure):
        if fig is not None:
            fig.show()

    def save(self, fig: go.Figure, filename: str):
        if filename.endswith(".html"):
            fig.write_html(filename)
        else:
            fig.write_image(filename)


viz = DecentralizationVisualizer()

---
# Part 1 — Foundations and Implications of Decentralization

## 1.1 The Importance of Decentralization

Decentralization underpins two foundational properties of public blockchains:
**property rights sovereignty** (tamper-proof, seizure-resistant ownership) and
**ledger assurance** (independent verifiability without trust assumptions).
This section provides contextual framing before quantitative analysis.

## 1.2 What Makes a Network Decentralized?

Four analytical dimensions: Auditability, Security, Governance, and Ownership & Distribution.
These dimensions map directly onto the quantitative metrics developed in Part 2.

## 1.3 Trade-offs in Decentralization

### 1.3.1 Scalability
Higher throughput and gas limits increase node hardware requirements,
concentrating participation. This section examines the scalability-decentralization tension
using Ethereum's gas limit debate as a case study.

### 1.3.2 Coordination
More decentralized networks face higher friction in governance upgrades
due to the need for broad stakeholder consensus.

In [ ]:
def fetch_chain_performance_metrics(chains: list = ["BTC", "ETH", "SOL"]) -> pd.DataFrame:
    """
    Fetch L1 performance comparison metrics: TPS, gas/fee levels, node count.
    Sources: Blockworks Research API, Artemis Analytics API (Phase 02)
    
    Returns:
        pd.DataFrame with columns: [chain, tps, avg_fee_usd, node_count, timestamp]
    """
    # TODO Phase 02: implement Blockworks / Artemis API call
    # Reference: https://app.blockworksresearch.com/analytics/l1
    # Reference: https://app.artemisanalytics.com/chains
    mock_data = {
        "chain": ["BTC", "ETH", "SOL"],
        "tps": [7, 15, 2000],
        "avg_fee_usd": [1.5, 2.0, 0.001],
        "node_count": [15000, 9000, 1900],
        "timestamp": [datetime.utcnow()] * 3
    }
    return pd.DataFrame(mock_data)

df_chain_perf = fetch_chain_performance_metrics()
fig = viz.bar_chart(df_chain_perf, title="L1 Performance Comparison: TPS by Chain",
                    x_col="chain", y_col="tps", color_col="chain")
viz.show(fig)

## 1.4 When is Decentralization Necessary?

Demand for decentralization is contextual. Risk-averse, institutional capital gravitates
toward battle-tested networks (BTC, ETH). Speculative and retail capital often favors
throughput over censorship resistance (SOL). This section compares on-chain activity
signals across the three assets.

In [ ]:
def fetch_transaction_count(chain: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch daily transaction count time series.
    Source: Glassnode API (BTC/ETH), Solscan or Helius API (SOL) — Phase 02
    
    Returns:
        pd.DataFrame with columns: [timestamp, tx_count, chain]
    """
    # TODO Phase 02: implement Glassnode /v1/transactions/count endpoint
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    mock = pd.DataFrame({
        "timestamp": dates,
        "tx_count": np.random.randint(300_000, 600_000, len(dates)),
        "chain": chain
    })
    return mock


def fetch_active_addresses(chain: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch daily active address count time series.
    Source: Glassnode API — Phase 02
    
    Returns:
        pd.DataFrame with columns: [timestamp, active_addresses, chain]
    """
    # TODO Phase 02: Glassnode /v1/addresses/active_count
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    mock = pd.DataFrame({
        "timestamp": dates,
        "active_addresses": np.random.randint(500_000, 1_200_000, len(dates)),
        "chain": chain
    })
    return mock


def fetch_rev(chain: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch daily Real Economic Value (REV) = fees + MEV (where applicable).
    Source: Blockworks Research API — Phase 02
    
    Returns:
        pd.DataFrame with columns: [timestamp, rev_usd, chain]
    """
    # TODO Phase 02: implement REV endpoint
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    mock = pd.DataFrame({
        "timestamp": dates,
        "rev_usd": np.random.uniform(1e6, 20e6, len(dates)),
        "chain": chain
    })
    return mock


START = "2023-01-01"
END = datetime.utcnow().strftime("%Y-%m-%d")

df_tx = pd.concat([fetch_transaction_count(c, START, END) for c in ["BTC", "ETH", "SOL"]])
df_addr = pd.concat([fetch_active_addresses(c, START, END) for c in ["BTC", "ETH", "SOL"]])
df_rev = pd.concat([fetch_rev(c, START, END) for c in ["BTC", "ETH", "SOL"]])

fig_tx = viz.line_chart(df_tx.pivot(index="timestamp", columns="chain", values="tx_count").reset_index(),
                        title="Daily Transaction Count: BTC vs ETH vs SOL",
                        x_col="timestamp", y_cols=["BTC", "ETH", "SOL"])
viz.show(fig_tx)

---
# Part 2 — Quantitative Measures of Decentralization

## 2.1 Node Participation and Accessibility

Node participation measures how accessible it is for independent parties to
run full validation infrastructure. Key dimensions include hardware cost,
disk requirements, bandwidth, and uptime. Lower cost = greater decentralization potential.

### 2.1.1 Cost to Run a Full Node

In [ ]:
# TODO Phase 02: fetch node cost estimates

In [ ]:
# TODO: visualization

### 2.1.2 Validator and Issuer Centralization

In [ ]:
# TODO Phase 02: fetch validator centralization data

In [ ]:
# TODO: visualization

### 2.1.3 Miner/Validator Reward Distribution (Entity-Level)

In [ ]:
# TODO Phase 02: fetch reward distribution data

In [ ]:
# TODO: visualization

### 2.1.4 Mining / Staking Pool Delegation Distribution

In [ ]:
# TODO Phase 02: fetch pool delegation distribution data

In [ ]:
# TODO: visualization

### 2.1.5 Miner/Validator Operating Cost vs. Annual Rewards

In [ ]:
# TODO Phase 02: fetch operating cost vs rewards data

In [ ]:
# TODO: visualization

### 2.1.6 Client Diversity

In [ ]:
# TODO Phase 02: fetch client diversity data

In [ ]:
# TODO: visualization

### 2.1.7 Entity-Level Stake Aggregation (Nakamoto Coefficient)

In [ ]:
# TODO Phase 02: fetch Nakamoto coefficient data

In [ ]:
# TODO: visualization

## 2.2 Governance Participation and Control

Governance decentralization measures whether protocol changes require
broad consensus or can be enacted unilaterally. Developer diversity is a
leading proxy for governance health — single-entity contributor dominance
indicates fragility.

### 2.2.1 Developer Contributions

In [ ]:
# TODO Phase 02: fetch developer contributions data

In [ ]:
# TODO: visualization

## 2.3 Ownership Distribution

Token ownership concentration is a structural risk factor. This section examines
supply distribution across address cohorts, founder/VC allocation vs. community issuance,
and the share of supply held by the largest addresses.

### 2.3.1 Token Supply by Address Cohort

In [ ]:
# TODO Phase 02: fetch supply by cohort data

In [ ]:
# TODO: visualization

### 2.3.2 Founder/VC Allocation vs. Community Issuance

In [ ]:
# TODO Phase 02: fetch token allocation breakdown data

In [ ]:
# TODO: visualization

### 2.3.3 % of Supply Held by Top X Addresses

In [ ]:
# TODO Phase 02: fetch top address supply concentration data

In [ ]:
# TODO: visualization

---
# Part 3 — BTC, ETH, and SOL Compared (and Conclusion)

## 3.1 Cross-Chain Comparative Summary

This section consolidates all decentralization vectors — node accessibility,
validator concentration, client diversity, governance, and ownership distribution —
into a side-by-side comparison of BTC, ETH, and SOL.

In [ ]:
# TODO Phase 02: build decentralization scorecard

In [ ]:
# TODO: visualization

## 3.2 Cost of Economic Attack

The economic security of a network can be proxied by the cost required
to execute a majority attack: 51% of hashpower for BTC, or acquiring
33% and 67% of staked supply for ETH and SOL.

In [ ]:
# TODO Phase 02: fetch attack cost estimates

In [ ]:
# TODO: visualization

## 3.3 Conclusion

[Narrative conclusion authored by David — to be written after quantitative analysis is complete.]

Key themes expected:
- BTC maintains the strongest structural decentralization case across node accessibility and immutability
- ETH trades some decentralization for programmability but preserves meaningful distribution
- SOL optimizes for performance at measurable cost to decentralization on most vectors
- The appropriate level of decentralization is contextual and capital-demand-driven